# 09 scenarios slope ramp — 勾配・上り下り・ランプ

**対象:** お客様（MPC 設計経験者）との **理論・数式・パラメータ** ディスカッション  
**Part 3/4** — Scenario 11–15

各シナリオは **路面 · 速度 · 勾配 · 実装** を結びつけています。  
理論の前提: [00_theory_grf_mpc_wbc.ipynb](./00_theory_grf_mpc_wbc.ipynb)  
QA 索引: [11_qa_discussion_master.ipynb](./11_qa_discussion_master.ipynb)

```bash
python scripts/scenario_labs.py --list
python scripts/scenario_labs.py --scenario sc11_bumpy_uphill_gravity
```


In [ ]:
import sys
from pathlib import Path

# mpc_dog ルートを sys.path に追加
ROOT = Path.cwd()
for p in [ROOT, *ROOT.parents]:
    if (p / "scripts" / "pympc_lab.py").exists():
        ROOT = p
        break
sys.path.insert(0, str(ROOT / "scripts"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from pympc_lab import (
    TUNING_GUIDE,
    apply_preset,
    compare_runs,
    load_param_study,
    load_preset_yaml,
    plot_friction_cone,
    run_flat_sim,
    run_speed_terrain_sim,
    run_speed_terrain_sim_resilient,
)

from tuning_labs import (
    TUNING_LABS,
    list_labs,
    run_lab,
    run_lab_pair,
    plot_speed_trial_journey,
    plot_param_study_mu,
    load_cached_lab_results,
)

%matplotlib inline
plt.rcParams["figure.figsize"] = (9, 4)
print(f"repo: {ROOT}")

from scenario_labs import (
    SCENARIO_LABS,
    compare_preset_table,
    run_scenario,
    run_scenario_pair,
    scenario_table,
)


## Scenario 11 — 凸凹上り坂 — 重力成分と pitch

| 項目 | 内容 |
|------|------|
| **ID** | `sc11_bumpy_uphill_gravity` |
| **分類** | slope / advanced |
| **路面** | bumpy_uphill（pitch +0.08 rad） |
| **速度** | 5.0 kph |
| **勾配** | uphill +0.08 rad |
| **preset** | `session04_bumpy_uphill` |

### シナリオ

上りでは $mg\sin\theta$ 分の追加 $F_x$ が必要。μ も freq も平坦より保守。

### 理論（Layer 2–3）

世界系 x 方向: $m a_x = \sum F_{ix} - mg\sin\theta$（近似）。

$$F_{ix}^{req} \approx m a_x + mg\sin\theta, \quad \theta \approx +0.08 \text{ rad}$$

### パラメータ焦点

| `mu` | 0.42→0.38 | `step_freq` | 1.20→1.10 | `speed_ramp_s` | 18→20 |

**実装:** `workshop_terrain.BUMPY_SCENES['bumpy_uphill']` pitch=+0.08

### ノウハウ

上り: μ↓ freq↓ ramp↑ ref_z やや↑。

### 議論用 Q&A

Q: pitch を MPC に入れている？
A: SRB+地形推定経由。明示 slope 項は sim 側重力投影。


In [ ]:
# resilient は時間がかかる → 短距離 A/B + Session 4 勝者キャッシュ参照
from scenario_labs import SCENARIO_BY_ID
from pympc_lab import apply_preset, compare_runs, load_speed_winners, run_speed_terrain_sim_resilient

sc = SCENARIO_BY_ID["sc11_bumpy_uphill_gravity"]
winners = load_speed_winners()
scene = sc.fail_kwargs.get("scene", "bumpy_flat")
if scene in winners:
    print("Session 4 winner cache:", winners[scene]["result"])

apply_preset(sc.preset)
fail_kw = dict(sc.fail_kwargs)
fail_kw["min_distance_m"] = min(8, fail_kw.get("min_distance_m", 8))
fail_kw["max_seconds"] = min(50, fail_kw.get("max_seconds", 50))
ok_kw = dict(sc.success_kwargs)
ok_kw["min_distance_m"] = min(10, ok_kw.get("min_distance_m", 10))
ok_kw["max_seconds"] = min(60, ok_kw.get("max_seconds", 60))
fail = run_speed_terrain_sim_resilient(**fail_kw)
ok = run_speed_terrain_sim_resilient(**ok_kw)
print(f"FAIL: {fail.get('distance_m', 0):.1f} m, falls={fail.get('falls')}")
print(f"OK:   {ok.get('distance_m', 0):.1f} m, falls={ok.get('falls')}")
fig = compare_runs([("FAIL", fail), ("OK", ok)])
plt.suptitle("Scenario 11: 凸凹上り坂 — 重力成分と pitch", y=1.02)
plt.show()


## Scenario 12 — 凸凹下り坂 — 最難（制動・支持）

| 項目 | 内容 |
|------|------|
| **ID** | `sc12_bumpy_downhill_brake` |
| **分類** | slope / expert |
| **路面** | bumpy_downhill（pitch -0.08 rad） |
| **速度** | 5.0 kph |
| **勾配** | downhill -0.08 rad |
| **preset** | `session04_bumpy_downhill` |

### シナリオ

下りは加速しやすく、duty=0.82, μ=0.35, ramp=22 s まで保守化が必要。

### 理論（Layer 2–3）

下り: $mg\sin|\theta|$ が加速方向。制動 $F_{ix}<0$ も摩擦円錐内で。

$$m a_x = \sum F_{ix} + mg\sin\theta, \quad \theta < 0 \text{（下り）}$$

### パラメータ焦点

| `duty_factor` | 0.76→0.82 | `mu` | 0.42→0.35 | `speed_ramp_s` | 22 |

**実装:** preset: `session04_bumpy_downhill.yaml`

### ノウハウ

下り最難。duty 最大級・μ 最小級・ramp 最長。

### 議論用 Q&A

Q: 下りで freq を上げて速く降りたい？
A: 支持不足で転倒増。duty↑ freq↓ が先。


In [ ]:
# resilient は時間がかかる → 短距離 A/B + Session 4 勝者キャッシュ参照
from scenario_labs import SCENARIO_BY_ID
from pympc_lab import apply_preset, compare_runs, load_speed_winners, run_speed_terrain_sim_resilient

sc = SCENARIO_BY_ID["sc12_bumpy_downhill_brake"]
winners = load_speed_winners()
scene = sc.fail_kwargs.get("scene", "bumpy_flat")
if scene in winners:
    print("Session 4 winner cache:", winners[scene]["result"])

apply_preset(sc.preset)
fail_kw = dict(sc.fail_kwargs)
fail_kw["min_distance_m"] = min(8, fail_kw.get("min_distance_m", 8))
fail_kw["max_seconds"] = min(50, fail_kw.get("max_seconds", 50))
ok_kw = dict(sc.success_kwargs)
ok_kw["min_distance_m"] = min(10, ok_kw.get("min_distance_m", 10))
ok_kw["max_seconds"] = min(60, ok_kw.get("max_seconds", 60))
fail = run_speed_terrain_sim_resilient(**fail_kw)
ok = run_speed_terrain_sim_resilient(**ok_kw)
print(f"FAIL: {fail.get('distance_m', 0):.1f} m, falls={fail.get('falls')}")
print(f"OK:   {ok.get('distance_m', 0):.1f} m, falls={ok.get('falls')}")
fig = compare_runs([("FAIL", fail), ("OK", ok)])
plt.suptitle("Scenario 12: 凸凹下り坂 — 最難（制動・支持）", y=1.02)
plt.show()


## Scenario 13 — 上りで下り preset — パラメータ持ち込み失敗（反対）

| 項目 | 内容 |
|------|------|
| **ID** | `sc13_uphill_wrong_preset` |
| **分類** | transition / advanced |
| **路面** | bumpy_uphill |
| **速度** | 5.0 kph |
| **勾配** | uphill（下り用 μ,duty 適用） |
| **preset** | `session04_speed_bumpy_base` |

### シナリオ

下り坂用の μ=0.35, duty=0.82 を上りに適用 → 加速不足・停滞（反対方向の誤適用）。

### 理論（Layer 2–3）

地形ごとに $\theta$ 符号が変わり、必要な $F_{ix}$ 分布も変化。preset は分離保存。

$$\text{uphill: } F_{ix}>0 \text{ 必要}, \quad \text{downhill preset: 過制動}$$

### パラメータ焦点

| 誤 | session04_bumpy_downhill の μ,duty | 正 | session04_bumpy_uphill |

**実装:** compare_presets: `session04_bumpy_downhill` vs `session04_bumpy_uphill`

### ノウハウ

YAML を地形別に。上り→下り切替時は全パラメータを見直し。

### 議論用 Q&A

Q: 上りから下りへ連続走行は？
A: 本 sim は scene 固定。実機は scene 検出+gain scheduling が必要。


In [ ]:
from scenario_labs import run_scenario_pair
from pympc_lab import compare_runs

pair = run_scenario_pair("sc13_uphill_wrong_preset")
fig = compare_runs(pair)
plt.suptitle("Scenario 13: 上りで下り preset — パラメータ持ち込み失敗（反対）", y=1.02)
plt.show()


## Scenario 14 — 下り duty 不足 — 支持時間が命

| 項目 | 内容 |
|------|------|
| **ID** | `sc14_downhill_duty_low` |
| **分類** | slope / expert |
| **路面** | bumpy_downhill |
| **速度** | 5.0 kph |
| **勾配** | downhill |
| **preset** | `session04_bumpy_downhill` |

### シナリオ

duty=0.70 では下り+凸凹で支持脚が足りず連続転倒。

### 理論（Layer 2–3）

duty↑ → $T_{stance}$↑ → 1 足あたり $F_{iz}$ 配分時間↑。

$$T_{stance} = duty / f_{step}, \quad F_{iz,i} \approx \frac{mg}{N_{stance}}$$

### パラメータ焦点

| `duty_factor` | 0.70 → 0.82 | 下り必須 |

**実装:** duty_factor を単独で A/B

### ノウハウ

下りで duty<0.78 は危険域。0.82 が S4 勝者値。

### 議論用 Q&A

Q: duty と freq のどちらが効く？
A: 下りは duty 優先。freq↓は MPC 解の時間的余裕。


In [ ]:
# resilient は時間がかかる → 短距離 A/B + Session 4 勝者キャッシュ参照
from scenario_labs import SCENARIO_BY_ID
from pympc_lab import apply_preset, compare_runs, load_speed_winners, run_speed_terrain_sim_resilient

sc = SCENARIO_BY_ID["sc14_downhill_duty_low"]
winners = load_speed_winners()
scene = sc.fail_kwargs.get("scene", "bumpy_flat")
if scene in winners:
    print("Session 4 winner cache:", winners[scene]["result"])

apply_preset(sc.preset)
fail_kw = dict(sc.fail_kwargs)
fail_kw["min_distance_m"] = min(8, fail_kw.get("min_distance_m", 8))
fail_kw["max_seconds"] = min(50, fail_kw.get("max_seconds", 50))
ok_kw = dict(sc.success_kwargs)
ok_kw["min_distance_m"] = min(10, ok_kw.get("min_distance_m", 10))
ok_kw["max_seconds"] = min(60, ok_kw.get("max_seconds", 60))
fail = run_speed_terrain_sim_resilient(**fail_kw)
ok = run_speed_terrain_sim_resilient(**ok_kw)
print(f"FAIL: {fail.get('distance_m', 0):.1f} m, falls={fail.get('falls')}")
print(f"OK:   {ok.get('distance_m', 0):.1f} m, falls={ok.get('falls')}")
fig = compare_runs([("FAIL", fail), ("OK", ok)])
plt.suptitle("Scenario 14: 下り duty 不足 — 支持時間が命", y=1.02)
plt.show()


## Scenario 15 — speed_ramp — 指令 $v^{ref}(t)$ の立ち上がり

| 項目 | 内容 |
|------|------|
| **ID** | `sc15_ramp_short_vs_long` |
| **分類** | speed / advanced |
| **路面** | bumpy_flat |
| **速度** | 5.0 kph |
| **勾配** | flat + 凸凹 |
| **preset** | `session04_speed_bumpy_base` |

### シナリオ

ramp=8 s では GRF 要求が急峻。18 s で mean_kph は下がるが距離到達。

### 理論（Layer 2–3）

Layer 1/指令: $v^{ref}(t)=v_t\min(t/T_r,1)$。急峻 → $|F_{ix}|$ スパイク。

$$\left|\frac{dv^{ref}}{dt}\right| \downarrow \Rightarrow \left|\sum F_{ix}\right| \text{ のピーク} \downarrow$$

### パラメータ焦点

| `speed_ramp_s` | 8 → 18 | 指令ランプ |

**実装:** `run_speed_terrain_sim` 内 `env._ref_base_lin_vel_H` を ramp

### ノウハウ

転倒→まず ramp↑。次に μ↓ freq↓。

### 議論用 Q&A

Q: ramp を無限に長くすれば良い？
A: 実用距離・時間制約あり。20 m / 5 kph なら 18–22 s が目安。


In [ ]:
from scenario_labs import run_scenario_pair
from pympc_lab import compare_runs

pair = run_scenario_pair("sc15_ramp_short_vs_long")
fig = compare_runs(pair)
plt.suptitle("Scenario 15: speed_ramp — 指令 $v^{ref}(t)$ の立ち上がり", y=1.02)
plt.show()


---

## Part 3 チェックリスト

- [ ] Scenario 11–15 それぞれ **数式 → パラメータ → 結果** を説明できる  
- [ ] fail / OK の差が **摩擦円錐 · gait · 指令 ramp** のどれか特定できる  
- [ ] `configs/pympc_presets/` の YAML と対応づけられる  

**次:** [10_scenarios_transition_limits.ipynb](./10_scenarios_transition_limits.ipynb)
